In [46]:
# ==============================================================================
# 🚀 NOTEBOOK DE FINE-TUNING PARA ANÁLISE DE SENTIMENTO - VERSÃO FINAL CORRIGIDA
# ==============================================================================
# OBJETIVO: Fazer o fine-tuning de um modelo de linguagem português usando
#           o dataset de reviews da Olist.
# ==============================================================================

# =========================================
# 🔧 PASSO 1: INSTALAÇÃO DAS BIBLIOTECAS
# =========================================
print("📦 Instalando bibliotecas necessárias...")
!pip install transformers datasets torch emoji evaluate accelerate -q
print("✅ Bibliotecas instaladas!")

# =========================================
# 📚 PASSO 2: IMPORTS E AUTENTICAÇÃO
# =========================================
import pandas as pd
import torch
import numpy as np
import evaluate
from datasets import Dataset, Features, Value, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from google.colab import userdata
from huggingface_hub import login

print("\n🔑 Realizando autenticação no Hugging Face...")
try:
    # O código procura por um segredo com o NOME 'HF_TOKEN' no Colab
    hf_token = userdata.get('messy')
    login(token=hf_token)
    print("✅ Autenticação bem-sucedida!")
except Exception as e:
    print(f"🛑 Falha na autenticação. Verifique se o segredo 'HF_TOKEN' foi criado corretamente.")
    print(e)

# =========================================
# 📊 PASSO 3: CARREGAMENTO E PREPARAÇÃO DO DATASET DA OLIST
# =========================================
print("\n🚚 Carregando e preparando o dataset da Olist...")
try:
    df = pd.read_csv('olist_order_reviews_dataset.csv')
    df = df.dropna(subset=['review_comment_message'])
    df = df[['review_comment_message', 'review_score']]

    def map_score_to_label(score):
        if score <= 2: return 0
        elif score == 3: return 1
        else: return 2

    df['label'] = df['review_score'].apply(map_score_to_label)
    df = df.rename(columns={'review_comment_message': 'text'})
    df = df[['text', 'label']]
    df.reset_index(drop=True, inplace=True)

    print(f"✅ Dataset da Olist carregado com {len(df)} exemplos.")
    print("\n📊 Distribuição das classes:")
    print(df['label'].value_counts(normalize=True))

except FileNotFoundError:
    print("\n🛑 ERRO: Arquivo 'olist_order_reviews_dataset.csv' não encontrado.")
    import sys
    sys.exit()

# =========================================
# 📂 PASSO 4: CONVERSÃO PARA DATASET HUGGINGFACE E DIVISÃO
# =========================================
print("\n🔄 Convertendo para o formato do Hugging Face e dividindo em treino/validação...")
features = Features({
    'text': Value('string'),
    'label': ClassLabel(num_classes=3, names=['NEGATIVO', 'NEUTRO', 'POSITIVO'])
})
dataset = Dataset.from_pandas(df, features=features)
train_test = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
ds_train = train_test["train"]
ds_val = train_test["test"]
print("✅ Dataset convertido e dividido!")

# =========================================
# 🤗 PASSO 5: CARREGAMENTO DO MODELO BASE E TOKENIZADOR
# =========================================
print("\n🧠 Carregando modelo pré-treinado e tokenizador...")
model_name = "pysentimiento/bertweet-pt-sentiment"
id2label = {0: "NEGATIVO", 1: "NEUTRO", 2: "POSITIVO"}
label2id = {"NEGATIVO": 0, "NEUTRO": 1, "POSITIVO": 2}

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
print("✅ Modelo e tokenizador carregados!")

# =========================================
# 🔠 PASSO 6: TOKENIZAÇÃO DOS DADOS
# =========================================
print("\n✍️  Tokenizando os datasets...")
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

ds_train = ds_train.map(tokenize_fn, batched=True)
ds_val = ds_val.map(tokenize_fn, batched=True)
ds_train.set_format("torch")
ds_val.set_format("torch")
print("✅ Tokenização completa!")

# =========================================
# 📈 PASSO 7: DEFINIÇÃO DAS MÉTRICAS DE AVALIAÇÃO
# =========================================
print("\n📉 Configurando métricas de avaliação (Acurácia, F1, Precisão, Recall)...")
metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="macro")
print("✅ Métricas configuradas!")

# =========================================
# ⚙️ PASSO 8: CONFIGURAÇÃO E EXECUÇÃO DO TREINAMENTO
# =========================================
print("\n⚙️ Configurando os argumentos de treinamento...")
training_args = TrainingArguments(
    output_dir="OlistSentimentModel",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=100,
    # --- CORREÇÃO APLICADA AQUI ---
    eval_strategy="steps", # O nome correto do argumento
    # -----------------------------
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

print("\n▶️  Iniciando o treinamento do modelo...")
trainer.train()
print("🏁 Treinamento concluído!")

# =========================================
# 🔮 PASSO 9: FUNÇÃO DE PREDIÇÃO E TESTE FINAL
# =========================================
print("\n🔬 Preparando para testar o modelo treinado...")
pipe = pipeline("sentiment-analysis", model=trainer.model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

print("\n🧪 Testando o modelo fine-tuned com novos exemplos:")
exemplos = [
    "A câmera é horrível, as fotos são borradas e sem cor. Odiei.",
    "O celular é excelente, a bateria dura dois dias! Superou minhas expectativas",
    "Nada de especial, um aparelho que apenas funciona. Razoável.",
    "Que celular rápido! O processador voa baixo e a tela é um espetáculo.",
    "A bateria demora pra carregar e descarrega com um sopro. Péssima compra."
]

for frase in exemplos:
    resultado = pipe(frase)[0]
    classe = resultado['label']
    prob = resultado['score']
    print(f"Texto: {frase}\n→ Sentimento: {classe} (confiança: {prob:.2%})\n")

📦 Instalando bibliotecas necessárias...
✅ Bibliotecas instaladas!

🔑 Realizando autenticação no Hugging Face...
✅ Autenticação bem-sucedida!

🚚 Carregando e preparando o dataset da Olist...
✅ Dataset da Olist carregado com 40977 exemplos.

📊 Distribuição das classes:
label
2    0.647436
0    0.265759
1    0.086805
Name: proportion, dtype: float64

🔄 Convertendo para o formato do Hugging Face e dividindo em treino/validação...
✅ Dataset convertido e dividido!

🧠 Carregando modelo pré-treinado e tokenizador...
✅ Modelo e tokenizador carregados!

✍️  Tokenizando os datasets...


Map:   0%|          | 0/32781 [00:00<?, ? examples/s]

Map:   0%|          | 0/8196 [00:00<?, ? examples/s]

✅ Tokenização completa!

📉 Configurando métricas de avaliação (Acurácia, F1, Precisão, Recall)...
✅ Métricas configuradas!

⚙️ Configurando os argumentos de treinamento...

▶️  Iniciando o treinamento do modelo...


/tmp/ipython-input-3614523773.py:144: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss


KeyboardInterrupt: 